# HCDE 530 — Week 5: Seattle coffee licenses — a first-look pass in pandas

This notebook runs the same checklist as the app-reviews demo, but on **`seattle_coffee_licenses.csv`**: open businesses whose trade name includes “coffee,” pulled from the City of Seattle’s public license data (Week 4).

You get one row per licensed business, with city, zip, license dates, and a NAICS industry label.

## Setup

In [1]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

## Load the dataset

The filename must match what is on disk in this folder: **`seattle_coffee_licenses.csv`**.

Ask the agent:
>"Load `seattle_coffee_licenses.csv` into a DataFrame called `df` using pandas. Print the shape and show the first few rows."

In [3]:
df = pd.read_csv('seattle_coffee_licenses.csv')
print(f"df: {df.shape[0]} rows, {df.shape[1]} columns")
df.head(10)

df: 263 rows, 6 columns


,trade_name,city,zip,license_start_date,expiration_date,naics_description
0,KAKAO COFFEE,SEATTLE,98109,20180101,NaN,Snack and Nonalcoholic Beverage Bars
1,70 & SUNNY COFFEE CO,SEATTLE,98121-1135,20251001,NaN,Snack and Nonalcoholic Beverage Bars
2,ACQUAINTANCE COFFEE,SEATTLE,98104-2024,20231210,NaN,Mobile Food Services
3,AFRIK GROCERY & COFFEE SHOP LLC,SEATTLE,98108,20150101,NaN,Convenience Retailers
4,ALCHEMY HARVEST COFFEE,SEATTLE,98126-3325,20250201,NaN,All Other Specialty Food Retailers
5,EAST COFFEE SHOP & MONEY MARKET,SEATTLE,98118,20130101,NaN,Supermarkets and Other Grocery Retailers (exce...
6,ALL CITY COFFEE COMPANY LLC,SEATTLE,98108,20010701,NaN,All Other Miscellaneous Retailers
7,ANCHORHEAD COFFEE,SEATTLE,98122-4415,20220210,NaN,Snack and Nonalcoholic Beverage Bars
8,ANCHORHEAD COFFEE,SEATTLE,98101-2288,20161001,NaN,All Other Specialty Food Retailers
9,ANCHORHEAD COFFEE,SEATTLE,98121-2161,20200808,NaN,Snack and Nonalcoholic Beverage Bars


The dataset has 263 rows and 6 columns. Each row is one licensed business with a coffee-related name. The expiration_date column is blank for every row, which is a known gap from the API and not random missing data.

## 1. What does your dataset look like? — `head()`

`head()` is the fastest way to see column names, data types in practice, and whether anything looks off (weird delimiters, blank columns, etc.).

Ask the agent:
>"Using `df`, show the first 5 rows with `df.head()`."

In [4]:
df.head()

,trade_name,city,zip,license_start_date,expiration_date,naics_description
0,KAKAO COFFEE,SEATTLE,98109,20180101,NaN,Snack and Nonalcoholic Beverage Bars
1,70 & SUNNY COFFEE CO,SEATTLE,98121-1135,20251001,NaN,Snack and Nonalcoholic Beverage Bars
2,ACQUAINTANCE COFFEE,SEATTLE,98104-2024,20231210,NaN,Mobile Food Services
3,AFRIK GROCERY & COFFEE SHOP LLC,SEATTLE,98108,20150101,NaN,Convenience Retailers
4,ALCHEMY HARVEST COFFEE,SEATTLE,98126-3325,20250201,NaN,All Other Specialty Food Retailers


The first 5 rows confirm the data looks clean with consistent city names, zip codes, and business categories with no obvious formatting issues.

## 2. `info()` — and the distribution of your most important column

`info()` lists each column, its dtype, and how many non-null values you have. For categorical work, the “most important” column is often the one that groups businesses into types — here, **`naics_description`** (the government industry category).

Ask the agent:
>"Run `df.info()`. Then show `value_counts()` for `naics_description` with `dropna=False` so missing labels are visible if any."

In [5]:
df.info()

df['naics_description'].value_counts(dropna=False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 263 entries, 0 to 262
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   trade_name          263 non-null    object 
 1   city                263 non-null    object 
 2   zip                 263 non-null    object 
 3   license_start_date  263 non-null    int64  
 4   expiration_date     0 non-null      float64
 5   naics_description   263 non-null    object 
dtypes: float64(1), int64(1), object(4)
memory usage: 12.5+ KB


naics_description
Snack and Nonalcoholic Beverage Bars                                                               84
Limited-Service Restaurants                                                                        60
Mobile Food Services                                                                               25
Coffee and Tea Manufacturing                                                                       22
All Other Specialty Food Retailers                                                                 16
Drinking Places (Alcoholic Beverages)                                                              11
Cafeterias, Grill Buffets, and Buffets                                                              7
Other Grocery and Related Products Merchant Wholesalers                                             4
Caterers                                                                                            4
Full-Service Restaurants                                        

Snack and Nonalcoholic Beverage Bars is the most common category with 84 businesses, followed by Limited-Service Restaurants with 60. This tells us most licensed coffee businesses in Seattle operate as casual cafe-style venues rather than full restaurants.

## 3. Filter to a meaningful subset — what’s in it?

The file includes coffee businesses outside Seattle city limits (e.g. Shoreline, Federal Way). A natural subset is **only businesses where `city` is `SEATTLE`** — same metro topic, tighter geography.

Ask the agent:
>"Create a new DataFrame `seattle_only` with only rows where `city` is SEATTLE (case-insensitive). Print how many rows that is, and show the first 10 rows."

In [6]:
seattle_only = df[df['city'].str.upper() == 'SEATTLE'].copy()
print(f"seattle_only: {len(seattle_only)} rows (of {len(df)} total)")
seattle_only.head(10)

seattle_only: 232 rows (of 263 total)


,trade_name,city,zip,license_start_date,expiration_date,naics_description
0,KAKAO COFFEE,SEATTLE,98109,20180101,NaN,Snack and Nonalcoholic Beverage Bars
1,70 & SUNNY COFFEE CO,SEATTLE,98121-1135,20251001,NaN,Snack and Nonalcoholic Beverage Bars
2,ACQUAINTANCE COFFEE,SEATTLE,98104-2024,20231210,NaN,Mobile Food Services
3,AFRIK GROCERY & COFFEE SHOP LLC,SEATTLE,98108,20150101,NaN,Convenience Retailers
4,ALCHEMY HARVEST COFFEE,SEATTLE,98126-3325,20250201,NaN,All Other Specialty Food Retailers
5,EAST COFFEE SHOP & MONEY MARKET,SEATTLE,98118,20130101,NaN,Supermarkets and Other Grocery Retailers (exce...
6,ALL CITY COFFEE COMPANY LLC,SEATTLE,98108,20010701,NaN,All Other Miscellaneous Retailers
7,ANCHORHEAD COFFEE,SEATTLE,98122-4415,20220210,NaN,Snack and Nonalcoholic Beverage Bars
8,ANCHORHEAD COFFEE,SEATTLE,98101-2288,20161001,NaN,All Other Specialty Food Retailers
9,ANCHORHEAD COFFEE,SEATTLE,98121-2161,20200808,NaN,Snack and Nonalcoholic Beverage Bars


232 out of 263 businesses are within Seattle city limits. The remaining 31 are in nearby cities like Shoreline and Federal Way, still relevant to the Seattle coffee scene but outside the city boundary.

## 4. Group by a category and find the average of a numeric column

`license_start_date` is stored as an integer like `20180101`. Parse it to a year, then group by **`naics_description`** and take the **mean start year** — a rough sense of which industry categories have newer vs. older license records in this slice.

Ask the agent:
>"Parse `license_start_date` to a year, group by `naics_description`, and show the mean year per category, sorted from highest to lowest. Round to 1 decimal place."

In [7]:
start_year = pd.to_datetime(
    df['license_start_date'].astype(str), format='%Y%m%d', errors='coerce'
).dt.year

df.assign(license_start_year=start_year).groupby('naics_description', dropna=False)['license_start_year'].mean().round(1).sort_values(ascending=False)

naics_description
Human Resources Consulting Services                                                                2025.0
All Other General Merchandise Retailers                                                            2024.7
Administrative Management and General Management Consulting Services                               2024.0
Caterers                                                                                           2024.0
Process, Physical Distribution, and Logistics Consulting Services                                  2024.0
Appliance Repair and Maintenance                                                                   2024.0
Other Similar Organizations (except Business, Professional, Labor, and Political Organizations)    2023.0
Mobile Food Services                                                                               2022.0
Baked Goods Retailers                                                                              2022.0
Cafeterias, Grill Buffets, a

Newer business categories like Mobile Food Services (avg 2022) and Caterers (avg 2024) have appeared more recently, while traditional coffee shops and Limited-Service Restaurants have older average start dates. This suggests mobile coffee carts are a growing format.

## 5. Where are the missing values? Are any columns entirely empty?

`isnull().sum()` counts missing cells per column. If a column is missing everywhere, you know it was not populated in the export (or the API did not return it) — not a random 5% missing pattern.

Ask the agent:
>"Show `df.isnull().sum()`. If any column has missing values, list which columns and how many."

In [8]:
missing = df.isnull().sum()
missing

trade_name              0
city                    0
zip                     0
license_start_date      0
expiration_date       263
naics_description       0
dtype: int64

Only expiration_date has missing values and all 263 are blank. This is a data coverage issue from the API and not random missingness. All other columns are complete and reliable for analysis.

### A note on this export

In the Week 4 script, **`expiration_date`** was not always returned by the API, so that column may be **entirely blank** in the CSV. That is a data *coverage* issue, not random row-level missingness — your analysis should not treat it like “sometimes we know the date.”

For a command-line version of the same analysis, see **`week5_seattle_coffee_licenses_analysis.py`** in this folder.